# Explore the raw Rice WC Hack data, straight from Box

No download, no Kaggle, no login. The Box share is public, so this notebook
pulls the raw `*-rice` files over the network and reads them in memory.

Runs on Google Colab as-is. Just run the cells top to bottom.

In [ ]:
import re, json, io, os, urllib.request
import pandas as pd

SHARE = "zfwy31xq4tbu6uaiglcsjeahvqo0vsav"   # the Rice WC Hack public share
BASE  = "https://rice.app.box.com"
UA    = {"User-Agent": "Mozilla/5.0"}

# Where cached copies live. On Colab this points into your Google Drive, so the
# cache survives the runtime being killed. Off Colab it just uses a local folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/ricehack_cache"
except ImportError:
    CACHE = os.path.expanduser("~/ricehack_cache")
os.makedirs(CACHE, exist_ok=True)
print("cache folder:", CACHE)


def _page(folder_id=None, page=1):
    url = f"{BASE}/s/{SHARE}" + (f"/folder/{folder_id}" if folder_id else "") + f"?page={page}"
    html = urllib.request.urlopen(urllib.request.Request(url, headers=UA)).read().decode("utf-8", "ignore")
    blob = re.search(r"Box\.postStreamData\s*=\s*(\{.*?\});", html, re.S).group(1)
    return json.loads(blob)["/app-api/enduserapp/shared-folder"]

def listing(folder_id=None):
    """All items in a shared folder, following Box's 20-per-page pagination."""
    d = _page(folder_id)
    out = list(d["items"])
    for p in range(2, d["pageCount"] + 1):
        out += _page(folder_id, p)["items"]
    return out

def datasets():
    return {i["name"]: i["id"] for i in listing() if i["type"] == "folder"}

def files(dataset):
    return sorted(listing(datasets()[dataset]), key=lambda i: i["name"])

def read(file_id, nrows=None):
    url = f"{BASE}/index.php?rm=box_download_shared_file&shared_name={SHARE}&file_id=f_{file_id}"
    raw = urllib.request.urlopen(urllib.request.Request(url, headers=UA)).read()
    return pd.read_csv(io.BytesIO(raw), compression="gzip", nrows=nrows, low_memory=False)


def load(dataset, n_files=None, refresh=False):
    """Load a whole dataset, caching it so you only ever download it once.

    The files inside a Box folder are NOT versions. They are arbitrary slices of
    one table from a parallel export, and they never overlap. Verified on
    daily-weather: all 31 slices together give exactly 401 stations x 1827 dates
    = 732,627 rows, zero duplicates. So you want all of them.

    First call downloads from Box and saves a .parquet copy (~75 s for weather).
    Every call after that reads the copy instead (~0.1 s), even after Colab
    restarts, because the copy sits in your Drive.
    """
    tag = dataset if n_files is None else f"{dataset}_first{n_files}"
    path = os.path.join(CACHE, f"{tag}.parquet")

    if os.path.exists(path) and not refresh:
        df = pd.read_parquet(path)
        print(f"loaded {len(df):,} rows from cache ({os.path.basename(path)})")
        return df

    chunks = files(dataset)
    picked = chunks if n_files is None else chunks[:n_files]
    if len(picked) < len(chunks):
        print(f"WARNING: reading {len(picked)} of {len(chunks)} slices "
              f"(~{100*len(picked)/len(chunks):.0f}% of the data)\n")

    parts = []
    for i, f in enumerate(picked, 1):
        print(f"  [{i}/{len(picked)}] {f['name']}")
        parts.append(read(f["id"]))
    df = pd.concat(parts, ignore_index=True)

    df.to_parquet(path, index=False)
    print(f"\ncached {len(df):,} rows -> {path}")
    return df

print("ready")

In [ ]:
# What's in the share?
for name in datasets():
    print(f"{name:36s} {len(files(name))} files")

In [ ]:
# ============================================================
# STEP 1 — cache the five datasets that fit in memory.
# Run this once. It takes roughly 15-25 minutes and saves each one to
# your Drive. After this, loading any of them takes under a second,
# forever, even if Colab restarts.
# ============================================================
SAFE = [
    "daily-weather-rice",               # tiny,   0.7 M rows
    "urban-heat-index-rice",            # small,  1.2 M rows
    "core-poi-geometry-rice",           # 0.7 GB, 0.5 M rows
    "daily-spend-brand-and-state-rice", # 1.9 GB,  17 M rows
    "spend-patterns-rice",              # 2.6 GB, 0.9 M rows
]

for name in SAFE:
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    d = load(name)
    del d          # free the memory; the Drive copy is what matters

print("\nAll five cached. Load any of them instantly with:  df = load('name')")

In [ ]:
# ============================================================
# STEP 2 — store-visits, the one that does not fit.
#
# Raw, it is ~223 million rows needing ~59 GB of RAM. Colab gives you 12.7 GB,
# so load() would crash. Two things keep it small:
#
#   1. Read fewer columns. The file has 13; we need 5. Measured: 1.84 GB per
#      slice down to ~0.8 GB.
#   2. Summarise, then throw the slice away. One slice is 6,971,860 daily rows,
#      which collapse to 98,975 monthly rows. That is ~70x smaller, and only
#      one slice is ever in memory at a time.
#
# So peak memory is about one slice, not the whole dataset.
# Takes ~30-40 minutes. You only ever run it once.
# ============================================================
import gc

# Every column here is used below. Adding an unused one costs real memory.
KEEP = ["MARKET", "CATEGORY", "NAICS_CODE", "LOCAL_DATE", "DAILY_VISITS"]
GROUP = ["MARKET", "CATEGORY", "NAICS_CODE", "month"]
out_path = os.path.join(CACHE, "store_visits_monthly.parquet")

if os.path.exists(out_path):
    visits = pd.read_parquet(out_path)
    print(f"loaded {len(visits):,} rows from cache")
else:
    running = []
    slices = files("store-visits-rice")
    for i, f in enumerate(slices, 1):
        url = f"{BASE}/index.php?rm=box_download_shared_file&shared_name={SHARE}&file_id=f_{f['id']}"
        raw = urllib.request.urlopen(urllib.request.Request(url, headers=UA)).read()
        part = pd.read_csv(io.BytesIO(raw), compression="gzip", usecols=KEEP, low_memory=False)

        part["month"] = part["LOCAL_DATE"].str.slice(0, 7)
        running.append(
            part.groupby(GROUP, as_index=False)
                .agg(visits=("DAILY_VISITS", "sum"), store_days=("DAILY_VISITS", "size"))
        )
        print(f"  [{i}/{len(slices)}] {len(part):,} daily rows -> {len(running[-1]):,} monthly")
        del raw, part           # drop the big frame before the next download
        gc.collect()

    # Each slice produced its own partial totals for the same months, so add them up.
    visits = (pd.concat(running, ignore_index=True)
                .groupby(GROUP, as_index=False)
                .agg(visits=("visits", "sum"), store_days=("store_days", "sum")))
    visits.to_parquet(out_path, index=False)
    print(f"\ncached {len(visits):,} monthly rows -> {out_path}")

visits.head()

In [ ]:
# ---- PICK ONE ----
# urban-heat-index-rice   core-poi-geometry-rice   daily-weather-rice
# store-visits-rice       spend-patterns-rice      daily-spend-brand-and-state-rice
DATASET = "daily-weather-rice"

# n_files=None reads every slice, which is what you want for real answers.
# The small folders (weather, urban-heat) take well under a minute.
# For store-visits, which is ~7 GB, pass a number like 3 for a first look.
df = load(DATASET)

print(f"\n{len(df):,} rows x {df.shape[1]} cols")
df.head()

In [ ]:
# Quick sanity pass: what is actually in each column?
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "pct_filled": (df.notna().mean() * 100).round(1),
    "n_unique": df.nunique(),
    "example": [str(df[c].dropna().iloc[0])[:50] if df[c].notna().any() else "" for c in df.columns],
})

In [ ]:
# The auto-report: a chart for every column, plus missing values and correlations.
!pip install -q ydata-profiling
from ydata_profiling import ProfileReport

ProfileReport(df, title=DATASET, minimal=True).to_notebook_iframe()

## Careful with `MARKET`

`MARKET` is a label, not a map boundary. Rows tagged with a US city can sit
anywhere from Hawaii to Alaska, and the POI data covers 23 countries. Run this
on anything with coordinates before you plot it.

## There is no store-level join between visits and places

`store-visits` and `core-poi-geometry` both have a column called `STORE_ID`, but
they hold different kinds of identifier, so they never match.

- `store-visits`: UUIDs, 100% of rows, all 36 characters
  (`3bdb3171-cb36-41b1-b684-c99b191fa733`)
- `core-poi-geometry`: short retailer codes on only 6.9% of rows
  (`06J1`, `2686`, `5250`)

Measured on the full places file against one visits slice: 30,640 place IDs
against 143,150 visit IDs, with **exactly 0 in common**.

So you cannot put an individual visited store on a map. The repo's
`DATA_TECH_DOCUMENT.md` calls this join "not reliable", which undersells it —
there is no join at all. Your options are to match loosely on brand name,
category and city, or to work at city level and accept that.

In [ ]:
if {"MARKET", "LATITUDE", "LONGITUDE"}.issubset(df.columns):
    g = df.groupby("MARKET").agg(
        n=("LATITUDE", "size"),
        lat_min=("LATITUDE", "min"), lat_max=("LATITUDE", "max"),
        lon_min=("LONGITUDE", "min"), lon_max=("LONGITUDE", "max"),
    ).round(2)
    g["lat_span"] = (g.lat_max - g.lat_min).round(1)
    display(g.sort_values("lat_span", ascending=False))
else:
    print("this dataset has no coordinates")